# Chapter 7: Simulation of Regulatory Networks

**Colab slideshow notebook**  
Based on Chapter 7 of the book "Computational Thinking for Life Scientists"
by Benny Chor and Amir Rubinstein.

## Chapter goals

By the end of this chapter, we want to understand:

- what a **computer simulation** (an *in-silico* experiment) is, and why it is useful in biology,
- continuous vs. **discrete** vs. **Boolean** models,
- a discrete model of regulatory networks: a **state graph** updated in "clock ticks",
- how to implement and run the model in Python,
- **attractors**: steady states and simulation cycles,
- two biological case studies: the yeast cell cycle and the circadian clock.

## Models and simulations

- A **computer simulation** imitates the behaviour of a system over time by running a program.
- It needs a **mathematical model**: a formal abstraction of the system that keeps the relevant aspects.
- **Continuous models** (e.g. differential equations) are detailed, but need many quantitative parameters
  (concentrations, kinetic constants) that are often unknown.
- **Discrete models** limit time and/or states to integers `0, 1, 2, …` — more intuitive, need less data,
  and are often faster to compute.
- **Boolean models** are the extreme case: every entity is just `0` (off) or `1` (on).

## 7.1 The general setting: a state graph

- **Nodes** = physical entities (mRNA, proteins, nutrients) or events (cell division, temperature change)
- **Edges** = regulation, with a **weight**: positive = activation, negative = inhibition
- Every node has a **state** in `0, 1, …, 9`:
  0 = absent / inactive, 9 = maximal expression or activity
- Initial states come from biological data; the simulation then advances in discrete steps `0, 1, 2, …`
- It ends when the network state stops changing, or is "trapped" in a cycle

## 7.1.2 A toy network

Three nodes: **A activates B**, **B activates C**, **C inhibits A**.

```
   A ──(+1)──▶ B ──(+1)──▶ C
   ▲                       │
   └─────────(−1)──────────┘
```

Initial network state: `(A, B, C) = (9, 0, 0)`.

The **effect** of node *i* on node *j* = (state of *i*) × (weight of the edge *i → j*).
For example, the effect of A on B is `9 × (+1) = +9`; the effect of C on A is `0 × (−1) = 0`.

## 7.1.3 The update rule

For each node, add up the effects of all its incoming edges (the **net effect**):

- net effect **> 0** → the state goes **up by 1**
- net effect **< 0** → the state goes **down by 1**
- net effect **= 0** → the state stays the same

All nodes are updated **at the same time** from the current network state (a *synchronous* model).

Step 0: `(9, 0, 0)` → step 1: `(9, 1, 0)` — B was activated by +9; A and C did not change.

**Exercise 1:** explain why the network state at step 2 is `(9, 2, 1)`.

## 7.1.3 The update rule
### 🤔 What will be the output?

In [ ]:
weights = [+1, -1, +1]      # effects of A, B, C on node X
states = [5, 9, 2]          # current states of A, B, C
net = sum(w * s for w, s in zip(weights, states))
print(net)
print("up" if net > 0 else "down" if net < 0 else "same")

## ✅ Answer
Why: net effect = 5 − 9 + 2 = −2 < 0, so the state of X goes **down** by 1.

In [ ]:
weights = [+1, -1, +1]      # effects of A, B, C on node X
states = [5, 9, 2]          # current states of A, B, C
net = sum(w * s for w, s in zip(weights, states))
print(net)
print("up" if net > 0 else "down" if net < 0 else "same")

-2
down


## 7.2 The model, formally

Nodes are numbered `0, 1, …, n−1`. The state of node *i* at step *t* is $s_i(t)$, in the range $0, \dots, U$.
The weight of the edge from *j* to *i* is $w(j, i)$ (0 if there is no edge).

The net effect on node *i* at step *t*:

$$\sigma_i(t) = \sum_{j=0}^{n-1} s_j(t) \cdot w(j, i)$$

The next state, kept inside the range $0 \dots U$:

$$
s_i(t+1) =
\begin{cases}
\min(U,\ s_i(t) + 1) & \text{if } \sigma_i(t) > 0 \\
\max(0,\ s_i(t) - 1) & \text{if } \sigma_i(t) < 0 \\
s_i(t) & \text{otherwise}
\end{cases}
$$

## 7.3.1 Representing the network in Python

A network is a **matrix of weights**: `G[i][j]` is the weight of the edge **from node i to node j**.

The toy network (rows and columns ordered A, B, C):

```python
G_toy = [[ 0, 1, 0],   # A -> B   (+1)
         [ 0, 0, 1],   # B -> C   (+1)
         [-1, 0, 0]]   # C -> A   (-1)
```

A **network state** is a list of the nodes' states, e.g. `[9, 0, 0]`.
A **simulation** is a list of network states, one per step.

In [ ]:
# Code snippet: the model's "engine" — update one node

def update_node(G, U, curr_states, i):
    ''' G - a network (weighted matrix, G[v][u] is the weight from v to u)
        U - maximal state
        curr_states - a list of the nodes' current states
        i - index of the node to update (between 0 and len(G)-1)
        apply the transition function to node i and return its new state
    '''
    n = len(curr_states)
    s = 0
    for j in range(n):
        s += G[j][i] * curr_states[j]   # effect of node j on node i

    if s > 0:
        new = min(U, curr_states[i] + 1)
    elif s < 0:
        new = max(0, curr_states[i] - 1)
    else:
        new = curr_states[i]
    return new

## 7.3.1 Representing the network in Python

In [ ]:
# Code snippet: one simulation step — update all nodes synchronously

def step(G, U, curr_states):
    ''' Return the list of states in the next time step.
        All nodes are updated from the SAME current states
        (synchronous model).'''
    n = len(curr_states)
    new_states = [None] * n
    for i in range(n):
        new_states[i] = update_node(G, U, curr_states, i)
    return new_states

G_toy = [[ 0, 1, 0],   # A
         [ 0, 0, 1],   # B
         [-1, 0, 0]]   # C
U = 9

print("step 0:", [9, 0, 0])
print("step 1:", step(G_toy, U, [9, 0, 0]))
print("step 2:", step(G_toy, U, step(G_toy, U, [9, 0, 0])))

## 7.3.1 Representing the network in Python

In [ ]:
# Code snippet: run the simulation until a steady state is reached
# (this version assumes the simulation does NOT end in a cycle)

def run(G, U, init_states):
    ''' Run the simulation until a steady state (two identical consecutive
        network states).
        Return the trajectory: a list of network states, one per step. '''
    trajectory = [init_states]
    curr_states = init_states
    next_states = step(G, U, curr_states)

    while next_states != curr_states:     # not a steady state yet
        trajectory += [next_states]
        curr_states = next_states
        next_states = step(G, U, next_states)

    trajectory += [next_states]            # the last two states are identical
    return trajectory


def pretty_print(trajectory):
    ''' print a simulation trajectory, one network state per line '''
    print("step:  network state:")
    for i in range(len(trajectory)):
        print(f"{i:4} : {trajectory[i]}")

pretty_print(run(G_toy, 9, [9, 0, 0]))

## One simulation step
### 🤔 What will be the output?

In [ ]:
G_toy = [[ 0, 1, 0],
         [ 0, 0, 1],
         [-1, 0, 0]]
print(step(G_toy, 9, [9, 0, 0]))
print(step(G_toy, 9, [9, 1, 0]))

## ✅ Answer
Why: A activates B, B activates C, C inhibits A — all nodes are updated from the **same** current states.

In [ ]:
G_toy = [[ 0, 1, 0],
         [ 0, 0, 1],
         [-1, 0, 0]]
print(step(G_toy, 9, [9, 0, 0]))
print(step(G_toy, 9, [9, 1, 0]))

[9, 1, 0]
[9, 2, 1]


## Steady state (fixed point)

From step 11 on, the network state is `(A, B, C) = (0, 9, 9)` and it no longer changes:
the simulation is "trapped" in a **steady state** (a *fixed point*).

It is easier to see the dynamics in a plot.

In [ ]:
# Code snippet: plot a trajectory — one line per node
import matplotlib.pyplot as plt

def plot_trajectory(trajectory, names, title=""):
    steps = range(len(trajectory))
    plt.figure(figsize=(8, 3.5))
    for i, name in enumerate(names):
        plt.plot(steps, [state[i] for state in trajectory], marker="o",
                 label=name)
    plt.xlabel("simulation step")
    plt.ylabel("node state")
    plt.title(title)
    plt.legend()
    plt.show()

plot_trajectory(run(G_toy, 9, [9, 0, 0]), ["A", "B", "C"],
                "Toy network, C starts at 0")
plot_trajectory(run(G_toy, 9, [9, 0, 5]), ["A", "B", "C"],
                "Toy network, C starts at 5")

## 7.1.4 Negative loops: modelling degradation

A **negative loop** is an edge from a node to itself with a negative weight.
It models natural **degradation**: the level of a substance decreases over time
even when nothing inhibits it.

We add self-inhibition (weight −1) to nodes B and C, and start from `(9, 0, 5)`:

In [ ]:
# Code snippet: toy network with self-inhibition on B and C

G_deg = [[ 0,  1,  0],   # A -> B
         [ 0, -1,  1],   # B -> B (degradation), B -> C
         [-1,  0, -1]]   # C -> A, C -> C (degradation)

print("update_node for A, B, C at [9, 0, 5]:",
      [update_node(G_deg, 9, [9, 0, 5], i) for i in range(3)])

trajectory = run(G_deg, 9, [9, 0, 5])
pretty_print(trajectory)
plot_trajectory(trajectory, ["A", "B", "C"], "With degradation of B and C")

## 7.1.4 Negative loops: modelling degradation
### 🤔 What will be the output?

In [ ]:
G_deg = [[ 0,  1,  0],
         [ 0, -1,  1],
         [-1,  0, -1]]
print([update_node(G_deg, 9, [0, 4, 0], i) for i in range(3)])

## ✅ Answer
Why: A is 0, so B gets only its own −1 (degradation) → 3; C gets +4 from B → 1; A gets 0 from C → stays 0.

In [ ]:
G_deg = [[ 0,  1,  0],
         [ 0, -1,  1],
         [-1,  0, -1]]
print([update_node(G_deg, 9, [0, 4, 0], i) for i in range(3)])

[0, 3, 1]


## Exercise 2

In the last simulation, the state of C **decreased** until step 3, then **increased** until step 6,
and then decreased again.

Explain why this happened. *Hint:* compute the net effect on C at a few steps.

## 7.1.5 Simulation cycles

A simulation ends in an **attractor**:

- a **steady state** — the same network state twice in a row, or
- a **cycle** — the network states repeat periodically, forever.

We add a **signal** node that constantly activates A (state 1, no incoming edges, so it never changes).

⚠️ Our `run` stops only at a steady state. On a cyclic network it would **never stop**.
So we use a version with a maximal number of steps.

In [ ]:
# Code snippet: toy network + a constant signal activating A

G_signal = [[ 0,  1,  0, 0],   # A
            [ 0, -1,  1, 0],   # B
            [-1,  0, -1, 0],   # C
            [ 1,  0,  0, 0]]   # signal -> A

def run_steps(G, U, init_states, max_steps):
    ''' run exactly max_steps simulation steps (no stopping condition) '''
    trajectory = [init_states]
    for _ in range(max_steps):
        trajectory += [step(G, U, trajectory[-1])]
    return trajectory

trajectory = run_steps(G_signal, 9, [9, 0, 5, 1], 40)
plot_trajectory(trajectory, ["A", "B", "C", "signal"],
                "A constant signal leads to a cycle")

## Exercises 4 and 5

Index the nodes of the last network as `0, 1, 2, 3` for A, B, C, signal, with initial state `[9, 0, 5, 1]`.

4. Compute by hand $\sigma_i(0)$ for every *i*.
5. Compute by hand $s_i(1)$ for every *i*.

Then check your answers with the next cell.

In [ ]:
# Code snippet: check Exercises 4 and 5

init = [9, 0, 5, 1]
sigma = [sum(init[j] * G_signal[j][i] for j in range(4)) for i in range(4)]
print("sigma_i(0) =", sigma)
print("s_i(1)     =", step(G_signal, 9, init))

## 7.3.3 Gathering attractor statistics

Which steady state does the network reach, over **all** combinations of initial states?

- 2 nodes with states 0…9 → 10² = 100 initial network states
- If most of them converge to the same steady state (a **main attractor**), the network is
  **robust**: it returns to the same outcome under different initial conditions.

`itertools.product` generates all the combinations:

In [ ]:
# Code snippet: all combinations of initial states with itertools.product
import itertools

combinations = list(itertools.product([0, 1, 2], repeat=2))
print("number of combinations:", len(combinations))
print("combinations:", combinations)

## 7.3.3 Gathering attractor statistics

In [ ]:
# Code snippet: what percent of initial states end in a given steady state?

def steady_state_size(G, U, steady_state):
    ''' % of initial state combinations that end up in the given steady
        state '''
    cnt = 0
    n = len(G)
    possible_init_states = [i for i in range(U + 1)]
    for init in itertools.product(possible_init_states, repeat=n):
        init = list(init)                    # convert tuple -> list
        trajectory = run(G, U, init)
        if trajectory[-1] == steady_state:   # steady state = attractor
            cnt += 1
    percent = 100 * cnt / (len(possible_init_states) ** n)
    return percent

print("toy network, % ending in [0, 9, 9]:",
      steady_state_size(G_toy, 9, [0, 9, 9]))
print("toy network, % ending in [0, 0, 0]:",
      steady_state_size(G_toy, 9, [0, 0, 0]))

## 7.3.3 Gathering attractor statistics
### 🤔 What will be the output?

In [ ]:
import itertools
combos = list(itertools.product([0, 1], repeat=3))
print(len(combos))
print(combos[:3])

## ✅ Answer
Why: 3 nodes with 2 possible states each give 2³ = 8 initial states, in this order.

In [ ]:
import itertools
combos = list(itertools.product([0, 1], repeat=3))
print(len(combos))
print(combos[:3])

8
[(0, 0, 0), (0, 0, 1), (0, 1, 0)]


## 7.4.1 Case study: the yeast cell cycle

Li et al. (2004) modelled key regulators of the cell cycle of budding yeast (*Saccharomyces cerevisiae*)
with a **Boolean** version of this model (`U = 1`).

- 11 nodes → 2¹¹ = 2048 possible initial states
- 86% of them reach the same steady state — the **main attractor**, which resembles the stationary G1 phase
- Conclusion: the yeast cell-cycle network is **robustly designed**

In [ ]:
# Code snippet: reproduce the yeast cell-cycle result (Li et al., 2004)

nodes = ["Cln3", "MBF", "SBF", "Cln1,2", "Cdh1", "Swi5", "Cdc20/Cdc14",
         "Clb5,6", "Sic1", "Clb1,2", "Mcm1/SFF"]

G_yeast = [
    [-1,  1,  1,  0,  0,  0,  0,  0,  0,  0,  0],
    [ 0,  0,  0,  0,  0,  0,  0,  1,  0,  0,  0],
    [ 0,  0,  0,  1,  0,  0,  0,  0,  0,  0,  0],
    [ 0,  0,  0, -1, -1,  0,  0,  0, -1,  0,  0],
    [ 0,  0,  0,  0,  0,  0,  0,  0,  0, -1,  0],
    [ 0,  0,  0,  0,  0, -1,  0,  0,  1,  0,  0],
    [ 0,  0,  0,  0,  1,  1, -1, -1,  1, -1,  0],
    [ 0,  0,  0,  0, -1,  0,  0,  0, -1,  1,  1],
    [ 0,  0,  0,  0,  0,  0,  0, -1,  0, -1,  0],
    [ 0, -1, -1,  0, -1, -1,  1,  0, -1,  0,  1],
    [ 0,  0,  0,  0,  0,  1,  1,  0,  0,  1, -1],
]

attractor = [0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0]
print(steady_state_size(G_yeast, 1, attractor),
      "% of initial vectors end at", attractor)
print("active nodes in the main attractor:",
      [nodes[i] for i in range(11) if attractor[i] == 1])

## Beyond Boolean: robustness with 10 levels

Rubinstein et al. (2007) re-examined the yeast network with `U = 9`.

- There are 10¹¹ initial network states — too many to try them all.
- Instead, initial states were **sampled at random**.
- 86.6% of the simulations converged to the main attractor (a G1-like state),
  which supports the robustness conclusion even under strong "noise" in the components' levels.

## 7.4.2 Case study: the circadian clock

- Rubinstein et al. (2016): a 22-node network of the main regulators of the vertebrate circadian clock.
- The simulation converged to a **cycle**: node states rise and fall periodically, as expected of a clock.
- Adding a **sunlight** node, connected in different hypothetical ways, produced "computational phase response curves".
  Only a few hypotheses matched the experimental curve — the model helped **narrow down hypotheses**.
- Mutations can be simulated too: a **knock-out** = delete a node or its outgoing edges;
  **over-expression** = raise its initial state.

## Reflection: why model and simulate?

- **Descriptive:** forces clarity and precision in describing a system or hypothesis
- **Analytic:** promotes understanding of the system and may give new insights
- **Predictive:** predicts behaviour under conditions not yet tested in the lab

Modelling is **iterative**: data → model → simulation → compare with experiments → adjust the model → new predictions → new experiments.

Simulation tools are often used as "black boxes". Knowing the model inside the box helps avoid misinterpreting results.

## Challenge Yourself — detecting simulation cycles (AI-assisted)

From this lesson on, **AI tools are allowed — and expected**. This is your first AI-assisted task.

**Task.** `run` stops only at a steady state. A simulation is **cyclic** if a network state appears
again (not necessarily in the next step).

(a) Write `run_with_cycles(G, U, init_states)` that stops at a steady state **or** at a cycle.
(b) Write `pretty_print_cycles(trajectory)` that also prints where the cycle begins and ends.

**What to submit:** the prompt you gave the AI tool, its first answer, the fixes you made,
and the tests that show your final version is correct. Read every line: you are responsible for the code.

In [ ]:
# Challenge: your (AI-assisted) solution here

def run_with_cycles(G, U, init_states):
    pass  # replace with your code

## Challenge Yourself — detecting simulation cycles (AI-assisted)

In [ ]:
# Code snippet: tests for run_with_cycles — from the book's example
# Run this cell after writing your function. Every assert must pass.

traj = run_with_cycles(G_signal, 9, [9, 0, 5, 1])

assert traj is not None, "run_with_cycles returned nothing"
assert len(traj) == 18, (f"expected 18 network states (steps 0..17), "
                         f"got {len(traj)}")
assert traj[17] == [1, 2, 3,
                    1], f"step 17 should be [1, 2, 3, 1], got {traj[17]}"
assert traj[8] == traj[17], ("the cycle should start at step 8 "
                              "and close at step 17")

# a network that ends in a steady state must still work as before
assert run_with_cycles(G_toy, 9, [9, 0, 0]) == run(G_toy, 9, [9, 0, 0])

print("all tests passed: the simulation ended in a cycle between steps 8 "
      "and 17")

## Chapter 7 Summary

In this chapter we studied:

- simulations as *in-silico* experiments, and continuous / discrete / Boolean models
- a discrete state-graph model: weighted activation and inhibition, states 0…U, synchronous steps
- the update rule $s_i(t+1)$ based on the net effect $\sigma_i(t)$
- an implementation in Python: `update_node`, `step`, `run`, `pretty_print`
- attractors: steady states and cycles; negative loops as degradation
- attractor statistics and robustness: the yeast cell cycle; the circadian clock